In [8]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

key = os.environ.get("GROQ_API_KEY")
print(f"Key loaded: {key[:8]}...{key[-4:]}")  # safely prints partial key
print(f"Starts with gsk_: {key.startswith('gsk_')}")  # Groq keys must start with gsk_

Key loaded: gsk_w9bH...Dx1J
Starts with gsk_: True


In [9]:
import re
import json
import os
import yfinance as yf
from crewai import Agent, Task, Crew, Process, LLM
from crewai_tools import CodeInterpreterTool, FileReadTool

In [10]:
from pydantic import BaseModel, Field

class QueryAnalysisOutput(BaseModel):
    """Structured output for the query analysis task."""
    symbol: str = Field(..., description="Stock ticker symbol (e.g., TSLA, AAPL).")
    timeframe: str = Field(..., description="Time period (e.g., '1d', '1mo', '1y').")
    action: str = Field(..., description="Action to be performed (e.g., 'fetch', 'plot').")

In [13]:
llm = LLM(model="groq/llama-3.1-8b-instant", api_key=os.environ.get("GROQ_API_KEY"))


# 1) Query parser agent
query_parser_agent = Agent(
    role="Stock Data Analyst",
    goal="Extract stock details and fetch required data from this user query: {query}.",
    backstory="You are a financial analyst specializing in stock market data retrieval.",
    llm=llm,
    verbose=True,
    memory=True,
)

query_parsing_task = Task(
    description="Analyze the user query and extract stock details.",
    expected_output="A dictionary with keys: 'symbol', 'timeframe', 'action'.",
    output_pydantic=QueryAnalysisOutput,
    agent=query_parser_agent,
)


# 2) Code writer agent
code_writer_agent = Agent(
    role="Senior Python Developer",
    goal="Write Python code to visualize stock data.",
    backstory="""You are a Senior Python developer specializing in stock market data visualization. 
                 You are also a Pandas, Matplotlib and yfinance library expert.
                 You are skilled at writing production-ready Python code""",
    llm=llm,
    verbose=True,
)

code_writer_task = Task(
    description="""Write Python code to visualize stock data based on the inputs from the stock analyst
                   where you would find stock symbol, timeframe and action.""",
    expected_output="A clean and executable Python script file (.py) for stock visualization.",
    agent=code_writer_agent,
)


# 3) Code interpreter agent (uses code interpreter tool from crewai)
code_interpreter_tool = CodeInterpreterTool()

code_execution_agent = Agent(
    role="Senior Code Execution Expert",
    goal="Review and execute the generated Python code by code writer agent to visualize stock data.",
    backstory="You are a code execution expert. You are skilled at executing Python code.",
    # tools=[code_interpreter_tool],
    allow_code_execution=True,   # This automatically adds the CodeInterpreterTool
    llm=llm,
    verbose=True,
)

code_execution_task = Task(
    description="""Review and execute the generated Python code by code writer agent to visualize stock data.""",
    expected_output="A clean and executable Python script file (.py) for stock visualization.",
    agent=code_execution_agent,
)

In [14]:
### --- CREW SETUP --- ###

crew = Crew(
    agents=[query_parser_agent, code_writer_agent, code_execution_agent],
    tasks=[query_parsing_task, code_writer_task, code_execution_task],
    process=Process.sequential
)

# Run the crew with an example query
result = crew.kickoff(inputs={"query": "Plot YTD stock gain of Tesla"})
print(result)

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Stock Data Analyst                                                                                      │
│                                                                                                                 │
│  Task: Analyze the user query and extract stock details.                                                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Stock Data Analyst                                                                                      │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  {"symbol":"Tesla","timeframe":"YTD","action":"plot"}                                                           │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


[2026-03-06 21:57:52][ERROR]: Failed to save to memory: Memory requires an LLM for analysis but initialization failed: Error importing native provider: OPENAI_API_KEY is required

To fix this, do one of the following:
  - Set OPENAI_API_KEY for the default model (gpt-4o-mini)
  - Pass a different model: Memory(llm="anthropic/claude-3-haiku-20240307")
  - Pass any LLM instance: Memory(llm=LLM(model="your-model"))
  - To skip LLM analysis, pass all fields explicitly to remember()
    and use depth="shallow" for recall.

Docs: https://docs.crewai.com/concepts/memory


╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Python Developer                                                                                 │
│                                                                                                                 │
│  Task: Write Python code to visualize stock data based on the inputs from the stock analyst                     │
│                     where you would find stock symbol, timeframe and action.                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Python Developer                                                                                 │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **stock_visualizer.py**                                                                                        │
│  ```python                                                                                                      │
│  """                                                                                                            │
│  This script visualizes stock data based on the inputs from the stock analyst.                                  │
│  """                                                                                                            │
│  import yfinance as yf                                                                                          │
│  import pandas as pd                                                                                            │
│  import matplotlib.pyplot as plt                                                                                │
│                                                                                                                 │
│  def get_stock_data(symbol, timeframe):                                                                         │
│      """                                                                                                        │
│      Retrieves stock data for the given symbol and timeframe.                                                   │
│                                                                                                                 │
│      Parameters:                                                                                                │
│      symbol (str): Stock symbol.                                                                                │
│      timeframe (str): Timeframe for the data (e.g., YTD, 1d, 5d, 1mo, 3mo, 6mo, 1y, 2y, 5y, 10y, ytd, max).     │
│                                                                                                                 │
│      Returns:                                                                                                   │
│      pd.DataFrame: Stock data.                                                                                  │
│      """                                                                                                        │
│      ticker = yf.Ticker(symbol)                                                                                 │
│      return ticker.history(period=timeframe)                                                                    │
│                                                                                                                 │
│  def plot_stock_data(data):                                                                                     │
│      """                                                                                                        │
│      Plots the stock data.                                                                                      │
│                                                                                                                 │
│      Parameters:                                                                                                │
│      data (pd.DataFrame): Stock data.                                                                           │
│      """                                               

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Code Execution Expert                                                                            │
│                                                                                                                 │
│  Task: Review and execute the generated Python code by code writer agent to visualize stock data.               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Code Execution Expert                                                                            │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Since the script provided is already in the correct format, my response will be to simply execute the code as  │
│  is.                                                                                                            │
│                                                                                                                 │
│  Here's the actual content of the script:                                                                       │
│                                                                                                                 │
│  ```python                                                                                                      │
│  """                                                                                                            │
│  This script visualizes stock data based on the inputs from the stock analyst.                                  │
│  """                                                                                                            │
│  import yfinance as yf                                                                                          │
│  import pandas as pd                                                                                            │
│  import matplotlib.pyplot as plt                                                                                │
│                                                                                                                 │
│  def get_stock_data(symbol, timeframe):                                                                         │
│      """                                                                                                        │
│      Retrieves stock data for the given symbol and timeframe.                                                   │
│                                                                                                                 │
│      Parameters:                                                                                                │
│      symbol (str): Stock symbol.                                                                                │
│      timeframe (str): Timeframe for the data (e.g., YTD, 1d, 5d, 1mo, 3mo, 6mo, 1y, 2y, 5y, 10y, ytd, max).     │
│                                                                                                                 │
│      Returns:                                                                                                   │
│      pd.DataFrame: Stock data.                                                                                  │
│      """                                                                                                        │
│      ticker = yf.Ticker(symbol)                                                                                 │
│      return ticker.history(period=timeframe)                                                                    │
│                                                                                                                 │
│  def plot_stock_data(data):                                                                                     │
│      """                                                                                                        │
│      Plots the stock data.                             

Since the script provided is already in the correct format, my response will be to simply execute the code as is. 

Here's the actual content of the script:

```python
"""
This script visualizes stock data based on the inputs from the stock analyst.
"""
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt

def get_stock_data(symbol, timeframe):
    """
    Retrieves stock data for the given symbol and timeframe.

    Parameters:
    symbol (str): Stock symbol.
    timeframe (str): Timeframe for the data (e.g., YTD, 1d, 5d, 1mo, 3mo, 6mo, 1y, 2y, 5y, 10y, ytd, max).

    Returns:
    pd.DataFrame: Stock data.
    """
    ticker = yf.Ticker(symbol)
    return ticker.history(period=timeframe)

def plot_stock_data(data):
    """
    Plots the stock data.

    Parameters:
    data (pd.DataFrame): Stock data.
    """
    plt.figure(figsize=(12, 6))
    data['Close'].plot()
    plt.title(f'{data.index[0].strftime("%Y-%m-%d")} to {data.index[-1].strftime("%Y-%m-%d")}')
   